In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
import time
import pandas as pd
import re
import os

chrome_driver_path = 'D:\Project\Liga 1 Indonesia\Liga-1-Indonesia\chromedriver-win64\chromedriver.exe'


# Setup Driver
options = webdriver.ChromeOptions()
#options.add_argument("--headless") # Biar Chrome-nya tidak dibuka (Jalan di belakang)
service = Service(chrome_driver_path)
driver = webdriver.Chrome(service=service, options=options)
driver.maximize_window() # Kalo headless, ini nggak berguna

folder_path = "Link Pemain"
csv_files = [file for file in os.listdir(folder_path) if file.endswith(".csv")]

for file in csv_files:
    file_path = os.path.join(folder_path, file)
    print(file)
    nama_klub = file.replace('.csv', '')

    data = pd.read_csv(file_path)
    print(file_path)
    link_pemain = data['Link Pemain'].tolist()
    data_pemain = []

    def get_xpath(xpath):
        try:
            return driver.find_element(By.XPATH, xpath).text.strip()
        except:
            return ""

    def extract_dynamic_sections():
        stats = {}
        try:
            all_sections = driver.find_elements(By.CLASS_NAME, "mt-15")
            for section in all_sections:
                try:
                    title = section.find_element(By.CLASS_NAME, "fs-14").text.strip()
                    items = section.find_elements(By.CLASS_NAME, "Stadium-item")
                    for item in items:
                        label_divs = item.find_elements(By.TAG_NAME, "div")
                        if len(label_divs) == 2:
                            label = label_divs[0].text.strip()
                            value = label_divs[1].text.strip()
                            stats[f"{title} - {label}"] = value
                except:
                    continue
        except:
            pass
        return stats

    for link in link_pemain[:3]:
        driver.get(link)
        time.sleep(2)
        # Data statis
        name = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[1]/div[1]/div/div[1]/span[1]')
        nation = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[1]/div[1]/div/div[2]/span')
        club = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[1]/div[1]/div/div[2]')
        height = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[2]/div[2]/div[1]/div[2]')
        weight = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[2]/div[2]/div[2]/div[2]')
        age = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[2]/div[2]/div[3]/div[2]')
        market_value = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[2]/div[2]/div[4]/div[2]')
        jersey_number = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[1]/div[2]/div[1]/div[2]')
        position = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[1]/div[2]/div[2]/div[2]')
        foot = get_xpath('/html/body/div/div/div/div[3]/div[2]/div[1]/div/div[1]/div[2]/div[3]/div[2]')

        # Data dinamis (statistik)
        dynamic_stats = extract_dynamic_sections()

        # Gabungkan semua data
        data = {
            "Name": name,
            "Nation": nation,
            "Club": club,
            "Height": height,
            "Weight": weight,
            "Age": age,
            "Market Value": market_value,
            "Jersey Number": jersey_number,
            "Position": position,
            "Foot": foot,
            "Link": link
        }

        # Tambahkan data statistik
        data.update(dynamic_stats)

        data_pemain.append(data)
        print(f"✅ Scraped: {name}")

    # ====================== SIMPAN CSV ======================

    df_pemain = pd.DataFrame(data_pemain)
    df_pemain.to_csv(f"Data Pemain/{nama_klub}-Pemain.csv", index=False, encoding="utf-8-sig")
    print(f"\n✅ Total pemain disimpan: {len(df_pemain)}")

driver.quit()

Dewa United Fc.csv
Link Pemain\Dewa United Fc.csv
✅ Scraped: Taisei Marukawa
✅ Scraped: Egy Maulana Vikri
✅ Scraped: Alex

✅ Total pemain disimpan: 3
Persib Bandung.csv
Link Pemain\Persib Bandung.csv
✅ Scraped: Beckham Nugraha
✅ Scraped: Mailson Lima Duarte Lopes
✅ Scraped: Ciro Alves

✅ Total pemain disimpan: 3
